### 1. Global Imports
We initialise the environment by importing standard libraries for filesystem manipulation (`os`, `shutil`, `pathlib`) and data randomization (`random`). These are essential for managing the large-scale image dataset efficiently without excessive memory overhead.

In [1]:
import os
import shutil
import random
from pathlib import Path

---
### 2. Workspace Directory Mapping
To maintain a clean project structure, we define three critical path variables:
* `base_path`: The original source directory containing raw DermNet data.
* `raw_data_path`: A temporary staging area for data consolidation.
* `final_split_path`: The final destination for the structured machine learning dataset.

In [2]:
base_path = Path('DermNet')
raw_data_path = Path('consolidated_data')
final_split_path = Path('dermal_vision_dataset')

---
### 3. Class Identification
The script dynamically scans the source directory to identify all diagnostic categories. This ensures that the 23 standard DermNet classes and our 2 custom safety classes (`Normal` and `No-Skin`) are included in the processing pipeline.

In [3]:
categories = [f.name for f in (base_path / 'train').iterdir() if f.is_dir()]

print(f"Detected {len(categories)} categories. Starting consolidation...")

Detected 25 categories. Starting consolidation...


---
### 4. Data Consolidation (Flattening)
Raw medical datasets are often split into arbitrary `train` and `test` folders upon download. In this step, we perform **Global Consolidation**:
* We merge all images for each category into a single master folder.
* We implement a renaming strategy (`prefixing`) to prevent filename collisions, ensuring no data is lost during the merger.

In [4]:
if not raw_data_path.exists():
    raw_data_path.mkdir()

for category in categories:
    cat_master_folder = raw_data_path / category
    cat_master_folder.mkdir(exist_ok=True)
    
    for folder_type in ['train', 'test']:
        source_dir = base_path / folder_type / category
        if source_dir.exists():
            for img in source_dir.glob('*'):
                if img.is_file():
                    shutil.copy(img, cat_master_folder / f"{folder_type}_{img.name}")

print("Consolidation complete. Starting 70/15/15 split...")

Consolidation complete. Starting 70/15/15 split...


---
### 5. Integrity Verification (Recursive Count)
Before proceeding to the split, we perform a recursive file count. This verification ensures that the total number of images in our consolidated staging area perfectly matches the count of the original source files, confirming that zero data leakage or loss occurred during creating the staging area.

In [9]:
def total_files(path):
    return sum(1 for _ in Path(path).rglob('*') if _.is_file())

# Grand Totals
print(f"Grand Total in Consolidated: {total_files(raw_data_path)}")
print(f"Grand Total in Original Split: {total_files(base_path)}")

Grand Total in Consolidated: 22057
Grand Total in Original Split: 22057


---
### 6. Detailed Category Audit
To ensure the model will be "foolproof," we perform a side-by-side category audit. This table verifies the exact image count for every disease class. Note: Any classes showing "EMPTY" indicate that the files have already been successfully moved forward to the final split phase.

In [8]:
base_counts = {}
for folder_type in ['train', 'test']:
    source_dir = base_path / folder_type
    if source_dir.exists():
        for cat_dir in source_dir.iterdir():
            if cat_dir.is_dir():
                count = len(list(cat_dir.glob('*')))
                base_counts[cat_dir.name] = base_counts.get(cat_dir.name, 0) + count

raw_counts = {}
if raw_data_path.exists():
    for cat_dir in raw_data_path.iterdir():
        if cat_dir.is_dir():
            raw_counts[cat_dir.name] = len(list(cat_dir.glob('*')))

print(f"{'Category':<35} | {'Base (Orig)':<12} | {'Raw (Consol)':<12} | {'Status'}")
print("-" * 75)

all_categories = sorted(set(base_counts.keys()) | set(raw_counts.keys()))

for cat in all_categories:
    orig = base_counts.get(cat, 0)
    raw = raw_counts.get(cat, 0)
    
    if orig == raw:
        status = "✅ MATCH"
    elif raw == 0 and orig > 0:
        status = "❌ EMPTY (Moved?)"
    else:
        status = f"⚠️ MISMATCH ({raw - orig:+})"
        
    print(f"{cat:<35} | {orig:<12} | {raw:<12} | {status}")

# 4. Grand Total Summary
print("-" * 75)
print(f"{'GRAND TOTAL':<35} | {sum(base_counts.values()):<12} | {sum(raw_counts.values()):<12} |")

Category                            | Base (Orig)  | Raw (Consol) | Status
---------------------------------------------------------------------------
Acne and Rosacea Photos             | 1150         | 1150         | ✅ MATCH
Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions | 1437         | 1437         | ✅ MATCH
Atopic Dermatitis Photos            | 612          | 612          | ✅ MATCH
Bullous Disease Photos              | 561          | 561          | ✅ MATCH
Cellulitis Impetigo and other Bacterial Infections | 361          | 361          | ✅ MATCH
Eczema Photos                       | 1544         | 1544         | ✅ MATCH
Exanthems and Drug Eruptions        | 505          | 505          | ✅ MATCH
Hair Loss Photos Alopecia and other Hair Diseases | 299          | 299          | ✅ MATCH
Herpes HPV and other STDs Photos    | 507          | 507          | ✅ MATCH
Light Diseases and Disorders of Pigmentation | 711          | 711          | ✅ MATCH
Lupus and other Conn

---
### 7. Professional Dataset Splitting (70/15/15)
In this final preparation step, we create the formal dataset architecture for the **EfficientNet-V2B0** model:
* **Shuffling**: A fixed seed (`42`) is used to thoroughly randomize images, breaking any clinical or temporal bias.
* **The 3-Way Split**: We divide the data into **Training (70%)**, **Validation (15%)**, and **Testing (15%)**.
* **Space-Efficiency**: We use `shutil.move` instead of `copy` to save disk space, re-indexing the files directly into the final clinical structure.

In [10]:
# --- STEP 2: SPLIT INTO TRAIN / VAL / TEST (SHUFFLE & MOVE VERSION) ---
split_ratios = {'train': 0.70, 'val': 0.15, 'test': 0.15}

for category in categories:
    # 1. Grab all images from the consolidated folder for this category
    images = list((raw_data_path / category).glob('*'))
    
    # 2. Shuffle to ensure Train/Val/Test sets are diverse and unbiased
    random.seed(42) # Fixed seed for reproducibility
    random.shuffle(images)
    
    # 3. Calculate split indices
    total_count = len(images)
    train_idx = int(total_count * split_ratios['train'])
    val_idx = int(total_count * (split_ratios['train'] + split_ratios['val']))
    
    # 4. Map the shuffled list to the three sets
    splits = {
        'train': images[:train_idx],
        'val': images[train_idx:val_idx],
        'test': images[val_idx:]
    }
    
    # 5. Move files (instead of copying) to save disk space
    for split_name, split_images in splits.items():
        target_dir = final_split_path / split_name / category
        target_dir.mkdir(parents=True, exist_ok=True)
        
        for img in split_images:
            # shutil.move re-indexes the file without using extra storage
            shutil.move(str(img), str(target_dir / img.name))

print(f"Success! Images shuffled and moved to: {final_split_path}")

Success! Images shuffled and moved to: dermal_vision_dataset


### Final Dataset Architecture Summary
The dataset is now structured for `tf.keras.utils.image_dataset_from_directory`. By including a dedicated **Validation** set (unseen during training but used for hyperparameter tuning) and a **Test** set (used only for the final evaluation), we ensure the reported accuracy of our **Skin Disease Predictor** model is scientifically robust and generalisable to real-world smartphone imagery.